In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader

In [2]:
file_path = 'C:/Users/mannu/2D Protien Folding/RS126.data.txt'
with open(file_path, 'r') as f:
    lines = f.readlines()

sequences, structures = [], []
for i in range(0, len(lines) - 1, 2):
    seq, struct = lines[i].strip(), lines[i+1].strip()
    if len(seq) == len(struct) and len(seq) > 0:
        sequences.append(seq)
        structures.append(struct)
window_size = 13
pad_length = window_size // 2
X_data, Y_data = [], []
for seq, struct in zip(sequences[:50], structures[:50]):
    padded_seq = ("X" * pad_length) + seq + ("X" * pad_length)
    for j in range(len(seq)):
        X_data.append(padded_seq[j : j + window_size])
        Y_data.append(struct[j])


In [3]:
(X_data[0:10], Y_data[0:10])

(['XXXXXXAPAFSVS',
  'XXXXXAPAFSVSP',
  'XXXXAPAFSVSPA',
  'XXXAPAFSVSPAS',
  'XXAPAFSVSPASG',
  'XAPAFSVSPASGA',
  'APAFSVSPASGAS',
  'PAFSVSPASGASD',
  'AFSVSPASGASDG',
  'FSVSPASGASDGQ'],
 ['C', 'C', 'E', 'E', 'E', 'E', 'E', 'C', 'C', 'C'])

In [4]:
alphabet = "ACDEFGHIKLMNPQRSTVWYX"
char_to_index = {char: idx for idx, char in enumerate(alphabet)}
vocab_size = len(alphabet) # 21
window_size = 13

# 1. Store raw integer IDs directly: Shape [Total_Rows, 13]
X_ints = torch.zeros(len(X_data), window_size, dtype=torch.long)

for row_idx, window in enumerate(X_data):
    for char_idx, char in enumerate(window):
        if char in char_to_index:
            X_ints[row_idx, char_idx] = char_to_index[char]

print(X_ints.shape)  # Should be [Total_Rows, 13]
# 2. Target labels (C, E, H -> 0, 1, 2)
shape_mapping = {'C': 0, 'E': 1, 'H': 2}
Y_ints = [shape_mapping[shape] for shape in Y_data]
Y_tensor = torch.tensor(Y_ints, dtype=torch.long)

# 3. Create DataLoader
dataset = TensorDataset(X_ints, Y_tensor)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"Data formatted for Embedding Layer!")
print(f"X_ints shape: {X_ints.shape}  <-- Simple 2D tensor [Batch, 13]")
print(f"Y_tensor shape: {Y_tensor.shape}")

torch.Size([8289, 13])
Data formatted for Embedding Layer!
X_ints shape: torch.Size([8289, 13])  <-- Simple 2D tensor [Batch, 13]
Y_tensor shape: torch.Size([8289])


In [5]:
class LearningBlock(nn.Module):
    def __init__(self, input_channels=256, dropout_rate=0.3):
        super(LearningBlock, self).__init__()
        self.linear1 = nn.Linear(input_channels, input_channels)  # 13*21 -> 256
        self.batch_norm1 = nn.BatchNorm1d(input_channels)
        self.Relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, x):
        x = self.linear1(x)
        x = self.batch_norm1(x)
        x = self.Relu(x)
        x = self.dropout(x)
        return x

In [6]:
class MiniFoldCNN(nn.Module):
    def __init__(self, vocab_size=21, input_channels=32, window_size=13, output_size=3, SequenceCount=2, hidden_dim=256):
            super(MiniFoldCNN, self).__init__()
            self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=input_channels)
            flattened_dim = 128 * window_size  # 128 * 13 = 1,664
            self.layers = nn.Sequential(
                # Feature Extractor (Conv1D)
                nn.Conv1d(in_channels=input_channels, out_channels=128, kernel_size=3, padding=1),    
                # Bridge 3D Conv grid -> 2D Flat vector [Batch, 1664]
                nn.Flatten(),
                nn.Linear(flattened_dim, hidden_dim),
                nn.ReLU(),
                # Stacking your LearningBlocks dynamically!
                *[LearningBlock(input_channels=hidden_dim, dropout_rate=0.3) for _ in range(SequenceCount)],
                # Final Classification Head (256 -> 3 classes: C, E, H)
                nn.Linear(hidden_dim, output_size)
            )

    def forward(self, x):
        x = self.embedding(x)
        x = x.transpose(1, 2)
        x = self.layers(x)
        return x

print("CNN Model successfully initialized!")
model = MiniFoldCNN(output_size=3, SequenceCount=4, hidden_dim=256)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

CNN Model successfully initialized!


In [9]:
NUM_EPOCHS = 100

print("--- STARTING CNN TRAINING ---")
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    
    for batch_X, batch_Y in train_loader:
        # batch_X is shape [64, 21, 13]
        
        # 1. Forward Pass
        outputs = model(batch_X)
        
        # 2. Calculate Loss
        loss = criterion(outputs, batch_Y)
        
        # 3. Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    if(epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Average Loss: {avg_loss}")

print("--- TRAINING COMPLETE ---")

--- STARTING CNN TRAINING ---
Epoch [1/100] | Average Loss: 0.003086467110268551
Epoch [10/100] | Average Loss: 8.802307465097888e-06
Epoch [20/100] | Average Loss: 2.5115343673904206e-06
Epoch [30/100] | Average Loss: 9.654746148813642e-07
Epoch [40/100] | Average Loss: 4.1669254686054887e-07
Epoch [50/100] | Average Loss: 1.9086376124022308e-07
Epoch [60/100] | Average Loss: 8.91745755255285e-08
Epoch [70/100] | Average Loss: 4.197548871393436e-08
Epoch [80/100] | Average Loss: 1.979787216473875e-08
Epoch [90/100] | Average Loss: 9.269372014397704e-09
Epoch [100/100] | Average Loss: 3.952801888371318e-09
--- TRAINING COMPLETE ---


In [11]:
def predict_protein_structure_embedding(protein_seq, trained_model, window_size=13, alphabet="ACDEFGHIKLMNPQRSTVWYX"):
    trained_model.eval() # Turn off Dropout
    
    pad_length = window_size // 2
    padded_seq = ("X" * pad_length) + protein_seq + ("X" * pad_length)
    
    char_to_idx = {char: i for i, char in enumerate(alphabet)}
    protein_len = len(protein_seq)
    
    # 1. Build 2D integer tensor: [Length of protein, 13]
    X_test_ints = torch.zeros(protein_len, window_size, dtype=torch.long)
    
    for i in range(protein_len):
        window = padded_seq[i : i + window_size]
        for char_idx, char in enumerate(window):
            if char in char_to_idx:
                X_test_ints[i, char_idx] = char_to_idx[char]
                
    # 2. Run Forward Pass
    with torch.no_grad():
        outputs = trained_model(X_test_ints) # Shape: [Length, 3]
        predicted_indices = torch.argmax(outputs, dim=1)
        
    # 3. Translate integers to letters
    int_to_shape = {0: 'C', 1: 'E', 2: 'H'}
    predicted_chars = [int_to_shape[int(idx.item())] for idx in predicted_indices]
    
    return "".join(predicted_chars)

# --- RUN INFERENCE TEST ---
test_input = "SIPPEVKFNKPFVFLMIEQNTKSPLFMGKVVNPTQK"
expected_output = "CCCCEEECCCCEEEEEEECCCCCEEEEEEECCCCCC"

pred = predict_protein_structure_embedding(test_input, model)

print("--- INFERENCE TEST ---")
print(f"Input:    {test_input}")
print(f"Expected: {expected_output}")
print(f"Pred:     {pred}")
print(f"Accuracy: {sum(p == e for p, e in zip(pred, expected_output)) / len(expected_output) * 100:.2f}%")

--- INFERENCE TEST ---
Input:    SIPPEVKFNKPFVFLMIEQNTKSPLFMGKVVNPTQK
Expected: CCCCEEECCCCEEEEEEECCCCCEEEEEEECCCCCC
Pred:     CCCCCCECCCCEEHEEEECCCCCCHEEEEEECCECE
Accuracy: 77.78%
